# CauseKit: identification-aware causal workflows

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Akanom/Causekit/blob/main/notebooks/kaggle/causekit_quickstart.ipynb)

[Run the published notebook on Kaggle](https://www.kaggle.com/code/akanom/causekit-quickstart)

This notebook is the shared Kaggle and Google Colab quickstart. It uses hash-pinned public data, fixed seeds, and explicit interpretation boundaries. Estimates are conditional on the declared design and assumptions; software output is not proof of exchangeability, exclusion, or parallel trends.

## Install the reviewed PyPI release

Install the exact CauseKit 0.7.0a6 release from the official PyPI index. The install has a three-minute timeout, does not clone the private development repository, and never requests a GitHub token.

In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys

EXPECTED_VERSION = "0.7.0a6"
PACKAGE = f"causekit[validation,plot,outputhub]=={EXPECTED_VERSION}"
try:
    installed_version = importlib.metadata.version("causekit")
except importlib.metadata.PackageNotFoundError:
    installed_version = None

if installed_version != EXPECTED_VERSION:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--disable-pip-version-check",
            "--index-url",
            "https://pypi.org/simple",
            PACKAGE,
        ],
        check=True,
        timeout=180,
    )
else:
    print(f"CauseKit {EXPECTED_VERSION} is already installed.")

for module_name in tuple(sys.modules):
    if module_name == "causekit" or module_name.startswith("causekit."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()

## Shared imports and deterministic configuration

In [ ]:
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:

    def display(value):
        print(value)


import causekit as ck
from causekit.datasets import REAL_DATASETS, load_real_dataset

SEED = 20_260_730
print("CauseKit version:", ck.__version__)
assert ck.__version__ == "0.7.0a6"

## 1. Load verified real data

CauseKit downloads only after explicit authorization, requires HTTPS, caches outside the repository, and verifies every file against the package registry's SHA-256 digest. The NSW data demonstrate randomized analysis, Cattaneo data demonstrate observational workflows, and the hospital data demonstrate DiD execution.

In [ ]:
nsw = load_real_dataset("nsw_mixtape", download=True)
cattaneo = load_real_dataset("cattaneo2", download=True)
hospital = load_real_dataset("hospdd", download=True)

for name, frame in {"nsw_mixtape": nsw, "cattaneo2": cattaneo, "hospdd": hospital}.items():
    print(name, len(frame), REAL_DATASETS[name].sha256)
    assert len(frame) > 0

## 2. Randomized treatment effect

Lin adjustment centers and fully interacts the declared pre-treatment covariates. Its causal interpretation relies on the NSW assignment design, consistency, no interference, and a defensible analysis population.

In [ ]:
baseline_columns = ["age", "educ", "black", "hisp", "marr", "nodegree", "re74", "re75"]
randomized = ck.RandomizedATE(adjustment="lin", covariance="robust").fit(
    nsw["re78"].astype(float),
    treatment=nsw["treat"].astype(int),
    covariates=nsw[baseline_columns].astype(float),
)
display(randomized.summary_frame().round(4))
display(pd.DataFrame([item.__dict__ for item in randomized.balance]).head().round(4))
assert np.isfinite(randomized.estimate)

## 3. Native causal ML on observational data

The partially linear DML path uses fold-local native ridge-GCV nuisances. The honest nonlinear R-learner keeps construction and evaluation rows immutable and reports held-out R-loss and calibration rather than unit-level confidence intervals. These estimates require conditional exchangeability, overlap, and valid nuisance conditions; cross-fitting does not remove unmeasured confounding.

In [ ]:
analysis = cattaneo.sample(n=2_000, random_state=SEED).sort_index()
covariate_columns = ["mmarried", "mage", "medu", "fbaby"]
X = analysis[covariate_columns].astype(float)
treatment = analysis["mbsmoke"].astype(int)
outcome = analysis["bweight"].astype(float)

dml = ck.PartiallyLinearDML(n_splits=5, random_state=SEED).fit(
    outcome, treatment=treatment, covariates=X
)
rlearner = ck.RLearner(
    n_splits=5,
    evaluation_fraction=0.4,
    random_state=SEED,
    calibration_groups=5,
    bootstrap_iterations=499,
).fit(outcome, treatment=treatment, covariates=X)

display(dml.summary_frame().round(4))
display(dml.nuisance_diagnostics.round(4))
display(rlearner.summary_frame().round(4))
display(
    pd.Series(
        {
            "honest_r_loss": rlearner.honest_r_loss,
            "constant_r_loss": rlearner.honest_constant_r_loss,
            "r_loss_gain": rlearner.r_loss_gain,
        }
    ).round(4)
)
display(rlearner.calibration_plot_data().round(4))
assert rlearner.construction_index.intersection(rlearner.evaluation_index).empty
assert rlearner.construction_nobs + rlearner.evaluation_nobs == len(analysis)

## 4. Matching point estimate with an estimated score

This compact demonstration estimates a full-sample Logit propensity and requests no matching standard error. CauseKit intentionally refuses to treat a generic or cross-fitted score as if it satisfied the narrow analytical first-step inference contract.

In [ ]:
import statsmodels.api as sm

logit_design = sm.add_constant(X, has_constant="add")
logit = sm.Logit(treatment.to_numpy(), logit_design).fit(disp=False)
propensity = pd.Series(logit.predict(logit_design), index=X.index)
matching = ck.NearestNeighborMatch(estimand="att", inference="none").fit(
    outcome,
    treatment=treatment,
    propensity=propensity,
    covariates=X,
    propensity_score_status="estimated",
    propensity_provenance="full_sample_statsmodels_logit_point_only",
)
display(matching.summary_frame().round(4))
display(matching.balance.round(4))
display(matching.balance_summary.round(4))
assert matching.inference == "none"

## 5. Conventional panel DiD

The hospital records are aggregated to a hospital-month panel. The conventional estimator remains the baseline; PT-All efficient DiD is a separately contracted, stronger-assumption path. The source describes these records as artificial, so this is an execution example rather than substantive empirical evidence.

In [ ]:
panel = (
    hospital.groupby(["hospital", "month"], as_index=False)
    .agg(outcome=("satis", "mean"), treated=("procedure", "max"))
    .sort_values(["hospital", "month"], kind="stable")
)
first_treated = panel.loc[panel["treated"].eq(1)].groupby("hospital")["month"].min()
panel["treatment_time"] = panel["hospital"].map(first_treated).fillna(np.inf)
did = ck.DifferenceInDifferences(control_group="never_treated").fit(
    panel,
    outcome="outcome",
    entity="hospital",
    time="month",
    treatment_time="treatment_time",
)
display(did.summary_frame().round(4))
display(did.event_study.round(4))
print("Pre-trend diagnostic:", did.pretrend)
assert np.isfinite(did.estimate)

## 6. Sharp regression discontinuity and graph

The deterministic cutoff design shows robust bias-corrected inference and the package-native plot surface. Continuity at the cutoff, no precise manipulation, and bandwidth credibility remain design assumptions.

In [ ]:
rng = np.random.default_rng(SEED)
running = pd.Series(np.r_[rng.uniform(-2.0, -0.02, 500), rng.uniform(0.02, 2.0, 500)])
rd_outcome = 1.0 + 0.6 * running + 1.4 * (running >= 0.0) + rng.normal(scale=0.5, size=len(running))
rd = ck.RegressionDiscontinuity(bandwidth=1.2, bias_bandwidth=1.6).fit(rd_outcome, running=running)
display(rd.summary_frame().round(4))
print(rd.bandwidth_selection)
print(rd.manipulation)
rd.plot()
assert np.isfinite(rd.bias_corrected_estimate)

## 7. Optional OutputHub report

Adapters transport already-fitted results and diagnostics; they do not refit estimators.

In [ ]:
from universal_output_hub import OutputHub

hub = OutputHub("CauseKit quickstart")
ck.add_to_outputhub(hub, randomized, name="NSW randomized effect")
ck.add_to_outputhub(hub, dml, name="Birthweight partially linear DML")
ck.add_to_outputhub(hub, did, name="Hospital conventional DiD")
print("OutputHub models:", len(hub.models))
print("OutputHub tables:", len(hub.tables))
assert len(hub.models) == 3

## Interpretation and reproducibility checklist

- Record the exact estimand, retained sample, treatment timing, nuisance roles, covariance target, and CauseKit version.
- Treat balance, overlap, first-stage, manipulation, and pre-trend output as diagnostics—not proofs of identification.
- Keep conventional DiD beside stronger PT-All sensitivity estimates.
- Keep honest R-learner evaluation rows out of every fit target and do not report unsupported unit-level intervals.
- Use the package's formal Python/R/Stata harnesses for parity claims; this cloud notebook is an executable adoption example.
- See the dedicated repository examples for Panel IV, survey/composition-robust repeated-section DiD, direct-ratio efficient DiD, and fuzzy RD.